# 需要予測② — 改善モデル：回帰と正直な検証をテストで測る

出典：第04章 03節／パス：`04_demand/02_improved.ipynb`／パッケージ：numpy・pandas・scikit-learn／実行時間の目安：約1分

このノート単体で実行できる。データ取得は①と同一であり、失敗時は `input/passengers.csv`、さらに欠品のときは内蔵生成へ切り替わる。
ベースラインのコードは `01_baseline.ipynb` に分けてある。正則化の選択は学習側の時系列分割で行い、最終評価はテストで1回だけ行う。


In [1]:
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

AIR_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"


def make_demand_series(n: int = 144, seed: int = 4) -> pd.DataFrame:
    r = np.random.default_rng(seed)
    t = np.arange(n)
    month = t % 12
    seas = 60.0 * np.sin(2 * np.pi * (month - 3) / 12.0) + 30.0 * (month == 6) + 40.0 * (month == 11)
    y = 120.0 + 1.1 * t + seas + r.normal(0, 12.0, size=n)
    dates = pd.date_range("1949-01", periods=n, freq="MS")
    return pd.DataFrame({"date": dates, "value": y})


try:
    raw = pd.read_csv(AIR_URL)
    raw.columns = ["date", "value"]
    raw["date"] = pd.to_datetime(raw["date"])
    ts = raw.sort_values("date").reset_index(drop=True)
    print("公開データを使用: 月次航空旅客数（1949-1960）")
except Exception:
    try:
        ts = pd.read_csv("input/passengers.csv", parse_dates=["date"])
        print("input/passengers.csv（合成代替）を使用")
    except FileNotFoundError:
        ts = make_demand_series()
        print("合成系列を生成（手順確認用）")
df = ts.copy()
df["lag1"] = df["value"].shift(1)
df["lag12"] = df["value"].shift(12)
df["trend"] = np.arange(len(df))
df["month"] = df["date"].dt.month.astype(str)
df = df.dropna().reset_index(drop=True)
X = df[["lag1", "lag12", "trend", "month"]]
yv = df["value"].to_numpy()
pre = ColumnTransformer([
    ("num", StandardScaler(), ["lag1", "lag12", "trend"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["month"]),
])
X_tr, X_te = X[:-24], X[-24:]
y_tr, y_te = yv[:-24], yv[-24:]
print(f"月数: {len(df)}（学習 {len(X_tr)}か月 / テスト {len(X_te)}か月）")


公開データを使用: 月次航空旅客数（1949-1960）
月数: 132（学習 108か月 / テスト 24か月）


In [2]:
# 改善：正則化の選択は学習側の時系列分割で行う（テストは最後の1回だけ）
import warnings
tss = TimeSeriesSplit(n_splits=3)
best = None
for a in [0.1, 1.0, 10.0, 100.0]:
    m = make_pipeline(pre, Ridge(alpha=a))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        s = -cross_val_score(m, X_tr, y_tr, cv=tss, scoring="neg_root_mean_squared_error")
    print(f"alpha={a:6.1f} / 時系列RMSE平均={s.mean():.2f}")
    if best is None or s.mean() < best[1]:
        best = (a, s.mean())
print(f"選択: alpha={best[0]}（{best[1]:.2f}）［学習側で選択］")
model = make_pipeline(pre, Ridge(alpha=best[0])).fit(X_tr, y_tr)
# 改善結果：ベースライン（前年同月）と改善モデルの差をテストで1回だけ測る
lag12_te = df["lag12"].to_numpy()[-24:]
rmse_b = float(np.sqrt(mean_squared_error(y_te, lag12_te)))
rmse_a = float(np.sqrt(mean_squared_error(y_te, model.predict(X_te))))
print(f"[改善モデル・Ridge] テストRMSE={rmse_a:.2f}［テストで1回］")
print(f"[改善結果] RMSE {rmse_b:.2f} → {rmse_a:.2f}（約{rmse_b / rmse_a:.1f}分の1）")
# 参考：ランダム分割と時系列分割の比較（学習側で測る。学習に使う月数も違う）
kf = KFold(n_splits=5, shuffle=True, random_state=0)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    s_kf = -cross_val_score(model, X_tr, y_tr, cv=kf, scoring="neg_root_mean_squared_error")
    s_ts = -cross_val_score(model, X_tr, y_tr, cv=tss, scoring="neg_root_mean_squared_error")
n_kf = [len(a) for a, _ in kf.split(X_tr)]
n_ts = [len(a) for a, _ in tss.split(X_tr)]
print(f"[ランダム5分割] RMSE平均={s_kf.mean():.2f}（学習 {min(n_kf)}〜{max(n_kf)}か月）")
print(f"[時系列3分割] RMSE平均={s_ts.mean():.2f}（学習 {'/'.join(map(str, n_ts))}か月）")
print(f"差（時系列−ランダム）={s_ts.mean() - s_kf.mean():+.2f}")

alpha=   0.1 / 時系列RMSE平均=16.28
alpha=   1.0 / 時系列RMSE平均=18.81
alpha=  10.0 / 時系列RMSE平均=21.03
alpha= 100.0 / 時系列RMSE平均=47.76
選択: alpha=0.1（16.28）［学習側で選択］
[改善モデル・Ridge] テストRMSE=19.76［テストで1回］
[改善結果] RMSE 49.99 → 19.76（約2.5分の1）
[ランダム5分割] RMSE平均=13.56（学習 86〜87か月）
[時系列3分割] RMSE平均=16.28（学習 27/54/81か月）
差（時系列−ランダム）=+2.72


In [3]:
# 残差点検：どこで外すかを見る（時系列分割の検証側の残差。学習に使っていない月で測る）
oof = []
for tr_idx, va_idx in tss.split(X_tr):
    m = make_pipeline(pre, Ridge(alpha=best[0])).fit(X_tr.iloc[tr_idx], y_tr[tr_idx])
    oof.append(pd.DataFrame({
        "month": df["month"].iloc[va_idx].astype(int).to_numpy(),
        "resid": y_tr[va_idx] - m.predict(X_tr.iloc[va_idx]),
    }))
oof = pd.concat(oof, ignore_index=True)
print(f"検証側の残差: {len(oof)}か月 / 平均={oof['resid'].mean():.2f} / 標準偏差={oof['resid'].std():.2f}")
worst = oof.groupby("month")["resid"].mean()
print("月別の平均残差（正＝実績が予測より多い）:")
print(worst.round(1).to_string())
assert np.isfinite(oof["resid"]).all()

検証側の残差: 81か月 / 平均=-1.27 / 標準偏差=16.55
月別の平均残差（正＝実績が予測より多い）:
month
1     -2.3
2    -17.0
3      5.5
4     -2.1
5      0.4
6     18.9
7     17.7
8      6.8
9    -15.9
10   -12.0
11   -13.8
12    -2.8


In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
_book_fig = Path("../../../books/python-machine-jitumu/assets/images")
FIG = str(_book_fig if _book_fig.is_dir() else Path("outputs"))
Path(FIG).mkdir(parents=True, exist_ok=True)
plt.figure()
plt.bar(["before (seasonal-naive)", "after (ridge)"], [rmse_b, rmse_a])
plt.ylabel("RMSE"); plt.title("RMSE before vs after (test)")
plt.savefig(f"{FIG}/code-result-j04-02.png", dpi=150, bbox_inches="tight"); plt.close()
print("saved code-result-j04-02.png")

saved code-result-j04-02.png


In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
_book_fig = Path("../../../books/python-machine-jitumu/assets/images")
FIG = str(_book_fig if _book_fig.is_dir() else Path("outputs"))
Path(FIG).mkdir(parents=True, exist_ok=True)
plt.figure(); plt.bar(worst.index.astype(str), worst.to_numpy())
plt.axhline(0, color="gray", linewidth=0.8)
plt.xlabel("month"); plt.ylabel("mean residual (validation folds)"); plt.title("residuals by month (time-series CV)")
plt.savefig(f"{FIG}/code-result-j04-03.png", dpi=150, bbox_inches="tight"); plt.close()
print("saved code-result-j04-03.png")

saved code-result-j04-03.png


## 読み取りどころ

- ランダム分割の方が良く見えること。未来の月が学習に混ざる分と、学習に使える月数が多い分の両方が効いている。
- 時系列の正直な測り方が `TimeSeriesSplit` であること。モデルの選択もこの測り方で行う。
- 残差は学習に使っていない月（検証側）で見ること。月別の偏りが「次の特徴量」のヒントになる。